# Process: timeseries → ACFs → INTs

For one dataset, this notebook registers and validates
the scans, loads and denoises the parcellated timeseries, regresses out task structure, and
computes autocorrelation functions (ACFs), intrinsic neural timescales (INTs) and timeseries
metrics, plus matched rest controls, length-matched rest and the bandpassed arm.

**Runs needed to reproduce the paper** — set `dataset`, `modalities` and `rest_variant` in the next cell:

| dataset | `modalities` | `rest_variant` | registry written | read by |
| --- | --- | --- | --- | --- |
| HCP-YA | `None` (3T) | `None` | `scan_registry_HCP-YA.csv` | fig2, sfig_rest_control, sfig_task_regression |
| HCP-YA | `['MRI3T', 'MRI7T']` | `None` | `scan_registry_HCP-YA_multi.csv` | fig1, sfig1 |
| HCP-D | `None` | `None` | `scan_registry_HCP-D.csv` | fig1, fig3, fig4, sfig_rest_control |
| WAND | `None` | `None` | `scan_registry_WAND.csv` | fig1 |
| Penn_LEAD | `None` | `None` | `scan_registry_Penn_LEAD.csv` | fig1 |

In [ ]:
import os
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np

from src.config import get_dataset_params
from src.data import DataLoader
from src.timescales import Timescales, rest_window_covered, rest_window_token

In [ ]:
# ── What to process ──────────────────────────────────────────────────────────────────────────
dataset = 'HCP-YA'      # 'HCP-YA', 'HCP-D', 'WAND' or 'Penn_LEAD'

# Overrides of the DATASET_PARAMS defaults in src/config.py. None keeps the default.
modalities   = None     # HCP-YA: None (3T) or ['MRI3T', 'MRI7T'] (the multi registry Figure 1 reads)
rest_variant = None     # HCP-YA 3T: 'noFIX' reads uncleaned rest and builds only the rest-derived outputs
                        # (raw rest, restlen-<task>, restctrl, restctrl-drift), tagged e.g.
                        # autocorr_noFIX_restctrl, in the unfiltered arm.
seg          = None     # parcellation, e.g. 'Schaefer10007'.

# ── Which stages to run ──────────────────────────────────────────────────────────────────────
# Every stage reads its cache when present. Can set force_recompute in a cell to rebuild.
RUN_TIMESCALES      = True    # ACFs and INTs, raw and task-residual (unfiltered arm)
RUN_METRICS         = True    # per-scan tSNR and variance
BUILD_REST_CONTROLS = True    # matched rest controls: restctrl, restctrl-drift
BUILD_REST_WINDOWS  = True    # length-matched rest: restlen-<task>
RUN_FILTERED        = True    # the 0.01-0.08 Hz bandpassed arm (HCP-YA and HCP-D only)
SAVE_TIMESERIES     = False   # optional export: per-task timeseries and residuals -> data/timeseries/
SAVE_TASK_CONTRASTS = False   # optional export: per-condition beta maps -> data/task_contrasts/

data_loader = DataLoader(dataset=dataset)

## Register, validate and load the timeseries

In [ ]:
params = get_dataset_params(dataset, seg=seg, rest_variant=rest_variant, modalities=modalities)
detrend             = params['detrend']
nuisance_regression = params['nuisance_regression']
task_regression     = params['task_regression']
rest_variant        = params['rest_variant']

subject_ids = data_loader.load_subject_ids(params['subject_ids_file'])
# subject_ids = subject_ids[:10]  # limit for testing

print(f"Dataset:    {dataset}")
print(f"Modalities: {params['modalities']}")
print(f"Sessions: {params['sessions']}")
print(f"Seg: {params['seg']}")
print(f"Detrend: {detrend}")
print(f"Nuis. regression: {nuisance_regression}")
print(f"Task regression: {task_regression}")
print(f"Rest source: {rest_variant}")

print(f"Subjects:   {len(subject_ids)}")
subject_ids[:5]

In [ ]:
for modality in params['modalities']:
    data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=params['sessions'],
        tasks=params['tasks'][modality],
        space=params['space'],
        seg=params['seg'],
        modality=modality,
        rest_variant=rest_variant,
    )

print(f"Registered {len(data_loader.scan_registry)} scans across {len(params['modalities'])} modality(ies).")

In [ ]:
data_loader.validate_scan_registry(
    verbose=True,
    remove_incomplete_subjects=params['remove_incomplete_subjects'],
    incomplete_unit=params.get('incomplete_unit', 'subject'),
)

In [ ]:
timeseries_data = data_loader.load_batch(
    detrend=detrend,
    nuisance_regression=nuisance_regression,
    frequency_filter=False,
    verbose=True,
)

In [ ]:
# The registry filename encodes the dataset, HCP-YA's modalities, a non-default parcellation and a non-default rest source, so no configuration overwrites another's registry.
modality_suffix = {('MRI3T',): '', ('MRI7T',): '_7T', ('MRI3T', 'MRI7T'): '_multi'}.get(
    tuple(params['modalities']), '_' + '_'.join(params['modalities']))
seg_suffix = f'_{seg}' if seg is not None else ''
rest_variant_tag = '' if rest_variant == 'clean' else f'_{rest_variant}'

registry_filename = f'scan_registry_{dataset}{modality_suffix}{seg_suffix}{rest_variant_tag}.csv'
data_loader.export_scan_registry(registry_filename)

scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['sub'].unique()
print(f'{len(subject_ids)} subjects in the validated registry')

## Task regression

In [ ]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=timeseries_data, verbose=True)
if task_regression:
    timescales_processor.regress_task_structure()

## Optional exports

Per-task timeseries (and task residuals) and per-condition beta maps, for use outside this
repository.

In [ ]:
if SAVE_TIMESERIES and rest_variant != 'clean':
    raise ValueError(f"timeseries dumps are written for the 'clean' rest source only; rest_variant is "
                     f"{rest_variant!r}, and its dumps would carry the same filenames.")
if SAVE_TIMESERIES:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'timeseries', dataset)
    os.makedirs(out_dir, exist_ok=True)

    tasks = sorted(set(
        part[len('task-'):]
        for k in timeseries_data
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_ts = {k: v for k, v in timeseries_data.items() if f'task-{task}_' in k}
        if not task_ts:
            continue
        out_path = os.path.join(out_dir, f"{dataset}_{params['seg']}_task-{task}_data-ts.npy")
        np.save(out_path, task_ts, allow_pickle=True)
        print(f"  {task}: {len(task_ts)} scans  →  {out_path}")

        if task_regression:
            task_resid = {k: v for k, v in timescales_processor.timeseries_residuals.items()
                          if f'task-{task}_' in k and v is not None}
            if task_resid:
                out_path = os.path.join(out_dir, f"{dataset}_{params['seg']}_task-{task}_data-tsresid.npy")
                np.save(out_path, task_resid, allow_pickle=True)
                print(f"  {task}: {len(task_resid)} residuals  →  {out_path}")

In [ ]:
if SAVE_TASK_CONTRASTS and task_regression:
    out_dir = os.path.join(data_loader.output_dir, 'data', 'task_contrasts', dataset)
    os.makedirs(out_dir, exist_ok=True)

    tasks = sorted(set(
        part[len('task-'):]
        for k in timescales_processor.task_contrasts
        for part in k.split('_')
        if part.startswith('task-')
    ))

    for task in tasks:
        task_data = {k: v for k, v in timescales_processor.task_contrasts.items()
                     if f'task-{task}_' in k and v is not None}
        if not task_data:
            continue
        out_path = os.path.join(out_dir, f"{dataset}_{params['seg']}_task-{task}_data-task_contrasts.npy")
        np.save(out_path, task_data, allow_pickle=True)
        print(f"  {task}: {len(task_data)} scans  →  {out_path}")

## Timeseries metrics

In [ ]:
if RUN_METRICS:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, force_recompute=False, verbose=True)

        # Task residuals do not depend on the rest source; the 'clean' run caches them.
        if task_regression and rest_variant == 'clean':
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, force_recompute=False, verbose=True)

## ACFs and INTs

Every INT here is `auc_zero_crossing`: the area under the ACF up to its first zero crossing
(`src.timescales.INT_METHOD`, implemented in `snaplab_tools.timescales`). It is computed for the raw
scans and, where a task GLM exists, for the task residuals.

In [ ]:
if RUN_TIMESCALES:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, verbose=True)
    timescales_processor.compute_timescale_measures(use_residuals=False, force_recompute=False, verbose=True)

In [ ]:
if RUN_TIMESCALES and task_regression and rest_variant == 'clean':
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, verbose=True)
    timescales_processor.compute_timescale_measures(use_residuals=True, suffix='resids', force_recompute=False, verbose=True)

## Matched rest controls and length-matched rest

In [ ]:
rest_window_matches_filtered = {'HCP-YA': ('WM',)}

# Suffix per control kind
REST_CONTROL_SUFFIX = {'full': 'restctrl', 'drift': 'restctrl-drift'}

In [ ]:
if BUILD_REST_CONTROLS and task_regression:
    timescales_processor.build_rest_controls(use_fft=True, verbose=True)

    for control_kind, control_suffix in REST_CONTROL_SUFFIX.items():
        timescales_processor.compute_autocorrelation_function(
            use_fft=True, use_rest_control=control_kind, force_recompute=False, verbose=True)
        timescales_processor.compute_timescale_measures(
            suffix=control_suffix, force_recompute=False, verbose=False)
elif BUILD_REST_CONTROLS:
    print(f'Skipping matched rest controls: {dataset} has no task GLM, so no control is defined.')

# Length-matched rest (restlen-<task>): each covered rest run cut, with nothing projected, to one window per
# matched task at the controls' offset, keyed by the rest run so load_notebook_data(rest_length='matched') can swap it in.
if BUILD_REST_WINDOWS and any(rest_window_covered(dataset, m) for m in params['modalities']):
    timescales_processor.resolve_rest_windows(verbose=True)
    for rest_match in timescales_processor.rest_window_span:
        timescales_processor.compute_autocorrelation_function(
            use_fft=True, use_rest_window=rest_match, force_recompute=False, verbose=True)
        timescales_processor.compute_timescale_measures(
            suffix=rest_window_token(rest_match),
            force_recompute=False, verbose=False)
        # The window's variance, on the same series as its ACF: metric_var_restlen-<task>.
        timescales_processor.compute_timeseries_measures(
            method='var', use_rest_window=rest_match, force_recompute=False, verbose=False)

## Frequency-specific timescales

HCP-YA and HCP-D only: they are the only datasets loaded unfiltered. WAND and Penn LEAD were
already bandpassed at 0.01–0.08 Hz inside XCP-D.

In [ ]:
# The bandpassed arm is built only for HCP-YA and HCP-D with the clean rest source, and anything else is skipped with a message.
if RUN_FILTERED and (rest_variant != 'clean' or dataset not in ('HCP-YA', 'HCP-D')):
    print(f'Skipping the bandpassed arm: built for HCP-YA and HCP-D with the clean rest source only '
          f'(dataset={dataset!r}, rest_variant={rest_variant!r}).')
elif RUN_FILTERED:
    filter_configs = [
        {'lowpass': 0.08, 'highpass': 0.01},  # bandpass (standard)
        # {'lowpass': None, 'highpass': 0.01},  # highpass
        # {'lowpass': 0.08, 'highpass': None},  # lowpass
    ]

    for cfg in filter_configs:
        print(cfg)
        lowpass  = cfg['lowpass']
        highpass = cfg['highpass']

        if highpass is not None and lowpass is not None:
            suffix = f'bp-{highpass}-{lowpass}hz'
        elif highpass is not None:
            suffix = f'hp-{highpass}hz'
        elif lowpass is not None:
            suffix = f'lp-{lowpass}hz'

        # Load with filtering applied before task regression
        timeseries_data_filt = data_loader.load_batch(
            detrend=detrend,
            nuisance_regression=nuisance_regression,
            frequency_filter=True,
            lowpass=lowpass,
            highpass=highpass,
            verbose=True,
        )

        processor_filt = Timescales(dataset=dataset, timeseries_data=timeseries_data_filt, verbose=False)
        if task_regression:
            processor_filt.regress_task_structure()

        # lowpass/highpass are passed for output naming only; filtering was applied at load time.
        processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False,
                                                        lowpass=lowpass, highpass=highpass, verbose=False)
        processor_filt.compute_timescale_measures(use_residuals=False,
                                                  suffix=suffix, force_recompute=False, verbose=False)

        if task_regression:
            processor_filt.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False,
                                                            lowpass=lowpass, highpass=highpass, verbose=False)
            processor_filt.compute_timescale_measures(use_residuals=True,
                                                      suffix=f'{suffix}_resids', force_recompute=False, verbose=False)

        # Controls are built inside this loop from the bandpassed rest, because one built outside would be
        # unfiltered and nothing would raise.
        if task_regression and BUILD_REST_CONTROLS:
            processor_filt.build_rest_controls(use_fft=True, verbose=False)
            for control_kind, control_suffix in REST_CONTROL_SUFFIX.items():
                processor_filt.compute_autocorrelation_function(
                    use_fft=True, use_rest_control=control_kind,
                    lowpass=lowpass, highpass=highpass, force_recompute=False, verbose=False)
                processor_filt.compute_timescale_measures(
                    suffix=f'{suffix}_{control_suffix}',
                    force_recompute=False, verbose=False)

        # Length-matched rest, inside this loop for the same reason as the controls.
        if BUILD_REST_WINDOWS and any(rest_window_covered(dataset, m) for m in params['modalities']):
            processor_filt.resolve_rest_windows(verbose=False)
            for rest_match in rest_window_matches_filtered.get(dataset, processor_filt.rest_window_span):
                processor_filt.compute_autocorrelation_function(
                    use_fft=True, use_rest_window=rest_match,
                    lowpass=lowpass, highpass=highpass, force_recompute=False, verbose=False)
                processor_filt.compute_timescale_measures(
                    suffix=f'{suffix}_{rest_window_token(rest_match)}',
                    force_recompute=False, verbose=False)